In [2]:
FACTOR_ID = "OFFICIAL_XGB_V14_0001_PURE_REOPEN15_EXPANDING_D4"


def main(datasources, start_date, end_date):
    """Pure 13:00-13:14 order-book model with fixed historical training."""
    import numpy as np
    import pandas as pd
    import dai
    import xgboost as xgb

    TRAIN_START = "2019-01-01 00:00:00"
    TRAIN_END = "2023-12-31 23:59:59"
    FEATURES = [
        "log_mid_return_reopen15_std",
        "range_reopen15_max",
        "imbalance3_reopen15_std",
        "official_pressure3_signed_reopen15_std",
        "amount_reopen15_sum",
        "spread_reopen15_median",
        "bid_depth3_reopen15_min",
        "askbid1_reopen15_std",
        "official_pressure3_neg_reopen15_mean",
        "spread_reopen15_std",
        "range_reopen15_mean",
        "depth_slope_reopen15_mean",
        "depth1_reopen15_min",
        "volume_reopen15_share",
        "reopen15_trade_share",
        "depth3_reopen15_last",
        "bid_depth3_reopen15_max",
        "bidask1_spread_reopen15_vw_volume",
        "official_pressure3_neg_reopen15_last",
        "toxicity_reopen15",
        "micro_dislocation3_reopen15_mean",
        "official_pressure3_signed_reopen15_mean",
        "depth_slope_diff_diff_reopen15_sum",
        "imbalance1_reopen15_mean",
        "depth_slope_change_reopen15",
        "micro_dislocation3_reopen15_last",
        "depth_curvature_reopen15_mean",
        "queue_refill_asymmetry_reopen15",
        "micro_dislocation1_reopen15_last",
        "depth_curvature_diff_reopen15_sum",
        "official_pressure3_neg_reopen15_zlast",
        "depth3_refill_reopen15",
        "depth1_reopen15_max",
        "depth_slope_diff_reopen15_mean",
        "ask_depth3_reopen15_min",
        "official_pressure3_signed_reopen15_last",
        "volume_reopen15_sum",
        "trade_reopen15_sum",
    ]
    MODEL_FEATURES = [f"{column}_xs" for column in FEATURES]

    def build_daily(table, sd, ed, need_label=False):
        sql = f"""
        WITH raw AS (
            SELECT
                date_trunc('day', date)::DATE AS trading_day,
                date AS bar_time,
                instrument::string AS instrument,
                EXTRACT(HOUR FROM date) * 100 + EXTRACT(MINUTE FROM date) AS hhmm,
                CAST(high AS DOUBLE) AS high_px,
                CAST(low AS DOUBLE) AS low_px,
                CAST(close AS DOUBLE) AS close_px,
                GREATEST(COALESCE(CAST(amount AS DOUBLE), 0.0), 0.0) AS amount_step,
                GREATEST(COALESCE(CAST(volume AS DOUBLE), 0.0), 0.0) AS volume_step,
                GREATEST(COALESCE(CAST(deal_number AS DOUBLE), 0.0), 0.0) AS trade_step,
                CAST(ask_price1 AS DOUBLE) AS ap1,
                CAST(bid_price1 AS DOUBLE) AS bp1,
                COALESCE(CAST(ask_volume1 AS DOUBLE), 0.0) AS av1,
                COALESCE(CAST(ask_volume2 AS DOUBLE), 0.0) AS av2,
                COALESCE(CAST(ask_volume3 AS DOUBLE), 0.0) AS av3,
                COALESCE(CAST(bid_volume1 AS DOUBLE), 0.0) AS bv1,
                COALESCE(CAST(bid_volume2 AS DOUBLE), 0.0) AS bv2,
                COALESCE(CAST(bid_volume3 AS DOUBLE), 0.0) AS bv3
            FROM {table}
            WHERE close > 0
              AND EXTRACT(HOUR FROM date) * 100 + EXTRACT(MINUTE FROM date)
                  BETWEEN 930 AND 1500
        ), geometry AS (
            SELECT *,
                (ap1 + bp1) / 2.0 AS mid,
                bv1 + av1 AS depth1,
                bv1 + bv2 + bv3 AS bid3,
                av1 + av2 + av3 AS ask3,
                bv1 + bv2 + bv3 + av1 + av2 + av3 AS depth3,
                bv2 + bv3 + av2 + av3 AS far_depth,
                bv1 + 0.7408182207 * bv2 + 0.5488116361 * bv3 AS weighted_bid3,
                av1 + 0.7408182207 * av2 + 0.5488116361 * av3 AS weighted_ask3,
                ABS(ap1 - bp1) / NULLIF((ap1 + bp1) / 2.0, 0) AS spread,
                (high_px - low_px) / NULLIF(close_px, 0) AS bar_range,
                (bv1 + av1) / NULLIF(bv2 + bv3 + av2 + av3, 0) AS depth_slope,
                (bv1 + av1) - 0.5 * (bv2 + bv3 + av2 + av3) AS depth_slope_diff,
                (bv1 + av1) - 2.0 * (bv2 + av2) + (bv3 + av3) AS depth_curvature
            FROM raw
        ), state0 AS (
            SELECT *,
                (bv1 - av1) / NULLIF(depth1, 0) AS imbalance1,
                (bid3 - ask3) / NULLIF(depth3, 0) AS imbalance3,
                (weighted_bid3 - weighted_ask3)
                    / NULLIF(weighted_bid3 + weighted_ask3, 0) AS weighted_imbalance3,
                ((ap1 * bv1 + bp1 * av1) / NULLIF(depth1, 0) - mid)
                    / NULLIF(mid, 0) AS micro_dislocation1,
                ((ap1 * weighted_bid3 + bp1 * weighted_ask3)
                    / NULLIF(weighted_bid3 + weighted_ask3, 0) - mid)
                    / NULLIF(mid, 0) AS micro_dislocation3,
                LN(NULLIF(mid, 0) / NULLIF(LAG(mid) OVER (
                    PARTITION BY trading_day, instrument ORDER BY bar_time
                ), 0)) AS log_mid_return,
                depth_slope_diff - LAG(depth_slope_diff) OVER (
                    PARTITION BY trading_day, instrument ORDER BY bar_time
                ) AS depth_slope_diff_diff,
                depth_curvature - LAG(depth_curvature) OVER (
                    PARTITION BY trading_day, instrument ORDER BY bar_time
                ) AS depth_curvature_diff
            FROM geometry
        ), state AS (
            SELECT *,
                weighted_imbalance3 * ABS(weighted_imbalance3)
                    / NULLIF(SQRT(ABS(spread)) + 1e-8, 0) AS pressure_signed,
                weighted_imbalance3 * weighted_imbalance3
                    / NULLIF(SQRT(ABS(spread)) + 1e-8, 0) AS pressure,
                volume_step * SIGN(COALESCE(imbalance1, 0.0)) AS signed_volume_proxy
            FROM state0
        ), agg AS (
            SELECT
                trading_day AS date,
                instrument,
                ARG_MAX(close_px, bar_time) AS close,
                SUM(volume_step) AS volume_total,
                SUM(trade_step) AS trade_total,
                STDDEV_SAMP(CASE WHEN hhmm BETWEEN 1300 AND 1314
                    THEN log_mid_return END) AS log_mid_return_reopen15_std,
                MAX(CASE WHEN hhmm BETWEEN 1300 AND 1314
                    THEN bar_range END) AS range_reopen15_max,
                STDDEV_SAMP(CASE WHEN hhmm BETWEEN 1300 AND 1314
                    THEN imbalance3 END) AS imbalance3_reopen15_std,
                STDDEV_SAMP(CASE WHEN hhmm BETWEEN 1300 AND 1314
                    THEN pressure_signed END) AS official_pressure3_signed_reopen15_std,
                SUM(CASE WHEN hhmm BETWEEN 1300 AND 1314
                    THEN amount_step ELSE 0.0 END) AS amount_reopen15_sum,
                MEDIAN(CASE WHEN hhmm BETWEEN 1300 AND 1314
                    THEN spread END) AS spread_reopen15_median,
                MIN(CASE WHEN hhmm BETWEEN 1300 AND 1314
                    THEN bid3 END) AS bid_depth3_reopen15_min,
                STDDEV_SAMP(CASE WHEN hhmm BETWEEN 1300 AND 1314
                    THEN -imbalance1 END) AS askbid1_reopen15_std,
                AVG(CASE WHEN hhmm BETWEEN 1300 AND 1314
                    THEN -pressure END) AS official_pressure3_neg_reopen15_mean,
                STDDEV_SAMP(CASE WHEN hhmm BETWEEN 1300 AND 1314
                    THEN -pressure END) AS official_pressure3_neg_reopen15_std,
                STDDEV_SAMP(CASE WHEN hhmm BETWEEN 1300 AND 1314
                    THEN spread END) AS spread_reopen15_std,
                AVG(CASE WHEN hhmm BETWEEN 1300 AND 1314
                    THEN bar_range END) AS range_reopen15_mean,
                AVG(CASE WHEN hhmm BETWEEN 1300 AND 1314
                    THEN depth_slope END) AS depth_slope_reopen15_mean,
                MIN(CASE WHEN hhmm BETWEEN 1300 AND 1314
                    THEN depth1 END) AS depth1_reopen15_min,
                SUM(CASE WHEN hhmm BETWEEN 1300 AND 1314
                    THEN volume_step ELSE 0.0 END) AS volume_reopen15_sum,
                SUM(CASE WHEN hhmm BETWEEN 1300 AND 1314
                    THEN trade_step ELSE 0.0 END) AS trade_reopen15_sum,
                ARG_MAX(CASE WHEN hhmm BETWEEN 1300 AND 1314 THEN depth3 END,
                    CASE WHEN hhmm BETWEEN 1300 AND 1314 THEN bar_time END) AS depth3_reopen15_last,
                MAX(CASE WHEN hhmm BETWEEN 1300 AND 1314
                    THEN depth3 END) AS depth3_reopen15_max,
                MIN(CASE WHEN hhmm BETWEEN 1300 AND 1314
                    THEN depth3 END) AS depth3_reopen15_min,
                AVG(CASE WHEN hhmm BETWEEN 1300 AND 1314
                    THEN depth3 END) AS depth3_reopen15_mean,
                MAX(CASE WHEN hhmm BETWEEN 1300 AND 1314
                    THEN bid3 END) AS bid_depth3_reopen15_max,
                ARG_MAX(CASE WHEN hhmm BETWEEN 1300 AND 1314 THEN bid3 END,
                    CASE WHEN hhmm BETWEEN 1300 AND 1314 THEN bar_time END) AS bid_depth3_reopen15_last,
                MIN(CASE WHEN hhmm BETWEEN 1300 AND 1314
                    THEN ask3 END) AS ask_depth3_reopen15_min,
                ARG_MAX(CASE WHEN hhmm BETWEEN 1300 AND 1314 THEN ask3 END,
                    CASE WHEN hhmm BETWEEN 1300 AND 1314 THEN bar_time END) AS ask_depth3_reopen15_last,
                SUM(CASE WHEN hhmm BETWEEN 1300 AND 1314
                    THEN imbalance1 * spread * volume_step ELSE 0.0 END)
                    AS bidask1_spread_reopen15_x_volume_sum,
                ARG_MAX(CASE WHEN hhmm BETWEEN 1300 AND 1314 THEN -pressure END,
                    CASE WHEN hhmm BETWEEN 1300 AND 1314 THEN bar_time END)
                    AS official_pressure3_neg_reopen15_last,
                SUM(CASE WHEN hhmm BETWEEN 1300 AND 1314
                    THEN signed_volume_proxy ELSE 0.0 END) AS signed_volume_proxy_reopen15_sum,
                AVG(CASE WHEN hhmm BETWEEN 1300 AND 1314
                    THEN micro_dislocation3 END) AS micro_dislocation3_reopen15_mean,
                ARG_MAX(CASE WHEN hhmm BETWEEN 1300 AND 1314 THEN micro_dislocation3 END,
                    CASE WHEN hhmm BETWEEN 1300 AND 1314 THEN bar_time END)
                    AS micro_dislocation3_reopen15_last,
                AVG(CASE WHEN hhmm BETWEEN 1300 AND 1314
                    THEN pressure_signed END) AS official_pressure3_signed_reopen15_mean,
                ARG_MAX(CASE WHEN hhmm BETWEEN 1300 AND 1314 THEN pressure_signed END,
                    CASE WHEN hhmm BETWEEN 1300 AND 1314 THEN bar_time END)
                    AS official_pressure3_signed_reopen15_last,
                SUM(CASE WHEN hhmm BETWEEN 1300 AND 1314
                    THEN COALESCE(depth_slope_diff_diff, 0.0) ELSE 0.0 END)
                    AS depth_slope_diff_diff_reopen15_sum,
                AVG(CASE WHEN hhmm BETWEEN 1300 AND 1314
                    THEN imbalance1 END) AS imbalance1_reopen15_mean,
                AVG(CASE WHEN hhmm BETWEEN 1300 AND 1314
                    THEN depth_curvature END) AS depth_curvature_reopen15_mean,
                SUM(CASE WHEN hhmm BETWEEN 1300 AND 1314
                    THEN COALESCE(depth_curvature_diff, 0.0) ELSE 0.0 END)
                    AS depth_curvature_diff_reopen15_sum,
                ARG_MAX(CASE WHEN hhmm BETWEEN 1300 AND 1314 THEN micro_dislocation1 END,
                    CASE WHEN hhmm BETWEEN 1300 AND 1314 THEN bar_time END)
                    AS micro_dislocation1_reopen15_last,
                MAX(CASE WHEN hhmm BETWEEN 1300 AND 1314
                    THEN depth1 END) AS depth1_reopen15_max,
                AVG(CASE WHEN hhmm BETWEEN 1300 AND 1314
                    THEN depth_slope_diff END) AS depth_slope_diff_reopen15_mean
            FROM state
            GROUP BY trading_day, instrument
        )
        SELECT
            date,
            instrument,
            COALESCE(close, 0.0) AS close,
            COALESCE(log_mid_return_reopen15_std, 0.0) AS log_mid_return_reopen15_std,
            COALESCE(range_reopen15_max, 0.0) AS range_reopen15_max,
            COALESCE(imbalance3_reopen15_std, 0.0) AS imbalance3_reopen15_std,
            COALESCE(official_pressure3_signed_reopen15_std, 0.0)
                AS official_pressure3_signed_reopen15_std,
            COALESCE(amount_reopen15_sum, 0.0) AS amount_reopen15_sum,
            COALESCE(spread_reopen15_median, 0.0) AS spread_reopen15_median,
            COALESCE(bid_depth3_reopen15_min, 0.0) AS bid_depth3_reopen15_min,
            COALESCE(askbid1_reopen15_std, 0.0) AS askbid1_reopen15_std,
            COALESCE(official_pressure3_neg_reopen15_mean, 0.0)
                AS official_pressure3_neg_reopen15_mean,
            COALESCE(spread_reopen15_std, 0.0) AS spread_reopen15_std,
            COALESCE(range_reopen15_mean, 0.0) AS range_reopen15_mean,
            COALESCE(depth_slope_reopen15_mean, 0.0) AS depth_slope_reopen15_mean,
            COALESCE(depth1_reopen15_min, 0.0) AS depth1_reopen15_min,
            COALESCE(volume_reopen15_sum / NULLIF(volume_total, 0), 0.0)
                AS volume_reopen15_share,
            COALESCE(trade_reopen15_sum / NULLIF(trade_total, 0), 0.0)
                AS reopen15_trade_share,
            COALESCE(depth3_reopen15_last, 0.0) AS depth3_reopen15_last,
            COALESCE(bid_depth3_reopen15_max, 0.0) AS bid_depth3_reopen15_max,
            COALESCE(bidask1_spread_reopen15_x_volume_sum
                / NULLIF(volume_reopen15_sum, 0), 0.0)
                AS bidask1_spread_reopen15_vw_volume,
            COALESCE(official_pressure3_neg_reopen15_last, 0.0)
                AS official_pressure3_neg_reopen15_last,
            COALESCE(ABS(signed_volume_proxy_reopen15_sum)
                / NULLIF(volume_reopen15_sum, 0), 0.0) AS toxicity_reopen15,
            COALESCE(micro_dislocation3_reopen15_mean, 0.0)
                AS micro_dislocation3_reopen15_mean,
            COALESCE(official_pressure3_signed_reopen15_mean, 0.0)
                AS official_pressure3_signed_reopen15_mean,
            COALESCE(depth_slope_diff_diff_reopen15_sum, 0.0)
                AS depth_slope_diff_diff_reopen15_sum,
            COALESCE(imbalance1_reopen15_mean, 0.0) AS imbalance1_reopen15_mean,
            COALESCE(depth_slope_diff_diff_reopen15_sum
                / NULLIF(ABS(depth3_reopen15_mean) + 1.0, 0), 0.0)
                AS depth_slope_change_reopen15,
            COALESCE(micro_dislocation3_reopen15_last, 0.0)
                AS micro_dislocation3_reopen15_last,
            COALESCE(depth_curvature_reopen15_mean, 0.0)
                AS depth_curvature_reopen15_mean,
            COALESCE(((bid_depth3_reopen15_last - bid_depth3_reopen15_min)
                - (ask_depth3_reopen15_last - ask_depth3_reopen15_min))
                / NULLIF(depth3_reopen15_mean + 1.0, 0), 0.0)
                AS queue_refill_asymmetry_reopen15,
            COALESCE(micro_dislocation1_reopen15_last, 0.0)
                AS micro_dislocation1_reopen15_last,
            COALESCE(depth_curvature_diff_reopen15_sum, 0.0)
                AS depth_curvature_diff_reopen15_sum,
            COALESCE((official_pressure3_neg_reopen15_last
                - official_pressure3_neg_reopen15_mean)
                / NULLIF(official_pressure3_neg_reopen15_std, 0), 0.0)
                AS official_pressure3_neg_reopen15_zlast,
            COALESCE((depth3_reopen15_last - depth3_reopen15_min)
                / NULLIF(ABS(depth3_reopen15_max - depth3_reopen15_min) + 1.0, 0), 0.0)
                AS depth3_refill_reopen15,
            COALESCE(depth1_reopen15_max, 0.0) AS depth1_reopen15_max,
            COALESCE(depth_slope_diff_reopen15_mean, 0.0)
                AS depth_slope_diff_reopen15_mean,
            COALESCE(ask_depth3_reopen15_min, 0.0) AS ask_depth3_reopen15_min,
            COALESCE(official_pressure3_signed_reopen15_last, 0.0)
                AS official_pressure3_signed_reopen15_last,
            COALESCE(volume_reopen15_sum, 0.0) AS volume_reopen15_sum,
            COALESCE(trade_reopen15_sum, 0.0) AS trade_reopen15_sum
        FROM agg
        """
        frame = dai.query(
            sql,
            filters={"date": [sd, ed]},
            compression=True,
        ).df()
        if frame.empty:
            return frame
        frame["date"] = pd.to_datetime(frame["date"])
        frame["instrument"] = frame["instrument"].astype(str)
        for column in ["close", *FEATURES]:
            frame[column] = pd.to_numeric(frame[column], errors="coerce")
        frame = frame.replace([np.inf, -np.inf], np.nan)
        frame = frame.sort_values(["instrument", "date"], kind="mergesort")
        if need_label:
            frame["label"] = (
                frame.groupby("instrument", sort=False)["close"].shift(-1)
                / frame["close"]
                - 1.0
            )
        return frame.reset_index(drop=True)

    def cross_sectional_features(frame):
        ranked = (
            frame.groupby("date", sort=False)[FEATURES]
            .rank(pct=True, method="average")
            .sub(0.5)
        )
        ranked.columns = MODEL_FEATURES
        ranked = ranked.replace([np.inf, -np.inf], np.nan).fillna(0.0)
        return pd.concat([frame.reset_index(drop=True), ranked.reset_index(drop=True)], axis=1)

    train = build_daily(
        "bigalpha_2026_stock_bar1m",
        TRAIN_START,
        TRAIN_END,
        need_label=True,
    )
    train_pool = dai.query(
        "SELECT date, instrument FROM bigalpha_2026_instruments",
        filters={"date": [TRAIN_START, TRAIN_END]},
        compression=True,
    ).df()
    train_pool["date"] = pd.to_datetime(train_pool["date"])
    train_pool["instrument"] = train_pool["instrument"].astype(str)
    train = pd.merge(train, train_pool, on=["date", "instrument"], how="inner")
    train = train.dropna(subset=["label"])
    train = cross_sectional_features(train)
    train = train.sort_values(["date", "instrument"], kind="mergesort").reset_index(drop=True)

    label_rank = train.groupby("date", sort=False)["label"].rank(
        pct=True, method="average"
    )
    relevance = np.minimum((label_rank.fillna(0.5).to_numpy() * 10).astype("int32"), 9)
    qid = pd.factorize(train["date"], sort=False)[0]
    model = xgb.XGBRanker(
        objective="rank:pairwise",
        n_estimators=220,
        max_depth=4,
        learning_rate=0.05,
        min_child_weight=50,
        subsample=0.82,
        colsample_bytree=0.75,
        reg_alpha=1.0,
        reg_lambda=20.0,
        tree_method="hist",
        n_jobs=1,
        random_state=20260718,
    )
    model.fit(train[MODEL_FEATURES], relevance, qid=qid, verbose=False)

    test = build_daily(datasources["bar1m"], start_date, end_date, need_label=False)
    pool = dai.query(
        "SELECT date, instrument FROM bigalpha_2026_instruments",
        filters={"date": [start_date, end_date]},
        compression=True,
    ).df()
    pool["date"] = pd.to_datetime(pool["date"])
    pool["instrument"] = pool["instrument"].astype(str)
    if test.empty:
        pool["factor"] = 0.0
        return pool[["date", "instrument", "factor"]].reset_index(drop=True)

    test = cross_sectional_features(test)
    test["factor"] = model.predict(test[MODEL_FEATURES])
    result = pd.merge(
        pool,
        test[["date", "instrument", "factor"]],
        on=["date", "instrument"],
        how="left",
    )
    result["factor"] = pd.to_numeric(result["factor"], errors="coerce")
    result["factor"] = result["factor"].replace([np.inf, -np.inf], np.nan)
    result["factor"] = result.groupby("date", sort=False)["factor"].transform(
        lambda values: values.fillna(values.median())
    )
    result["factor"] = result["factor"].fillna(0.0)
    return result[["date", "instrument", "factor"]].reset_index(drop=True)


if __name__ == "__main__":
    from bigmodule import M
    import dai

    datasources = {
        "bar1m": "bigalpha_2026_stock_bar1m",
        "financial": "bigalpha_2026_financial",
    }
    start_date = "2024-01-01 00:00:00"
    end_date = "2024-12-31 23:59:59"
    factor_data = main(datasources, start_date, end_date)
    factor_pool = dai.query(
        "SELECT * FROM bigalpha_2026_factorlib",
        filters={"date": [start_date, end_date]},
        compression=True,
    ).df()
    result = M.bigalpha_eval._latest(
        factor_data=factor_data,
        factor_pool=factor_pool,
        start_date=start_date,
        end_date=end_date,
        process_pools=False,
        show=True,
    )
